# Smart Hospitality Management System
## Module 1: Hotel Booking Prediction

End-to-end pipeline: cleaning -> EDA -> feature engineering -> preprocessing -> modeling (baseline ML + Deep Learning ANN).

## 1. Data Cleaning

In [3]:
"""
Module 1: Hotel Booking Prediction
Step 1: Data Cleaning
"""
import pandas as pd
import numpy as np

df = pd.read_csv('dataset/hotel_booking.csv')
print(f"Original shape: {df.shape}")

# ------------------------------------------------------------------
# 1. Drop target-leakage & PII columns
# ------------------------------------------------------------------
# reservation_status / reservation_status_date directly encode the outcome
# (e.g. status='Canceled' <=> is_canceled=1) -> leakage, must drop.
# name/email/phone-number/credit_card are synthetic PII with zero predictive value.
leak_and_pii = ['reservation_status', 'reservation_status_date',
                 'name', 'email', 'phone-number', 'credit_card']
df = df.drop(columns=leak_and_pii)

# ------------------------------------------------------------------
# 2. Handle missing values
# ------------------------------------------------------------------
print("\nMissing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# company: ~94% missing -> not usable as a feature directly, drop column
df = df.drop(columns=['company'])

# agent: missing means "no agent involved" -> flag + fill
df['has_agent'] = df['agent'].notnull().astype(int)
df = df.drop(columns=['agent'])

# country: fill missing with 'Unknown'
df['country'] = df['country'].fillna('Unknown')

# children: 4 missing -> fill with 0 (most common value, and NaN likely means none)
df['children'] = df['children'].fillna(0).astype(int)

# ------------------------------------------------------------------
# 3. Fix data errors / invalid rows
# ------------------------------------------------------------------
# adr (average daily rate) should not be negative
before = len(df)
df = df[df['adr'] >= 0]
print(f"\nDropped {before - len(df)} rows with negative adr")

# cap extreme adr outlier(s) - one row at 5400 vs 75th pct ~126
before = len(df)
df = df[df['adr'] < df['adr'].quantile(0.999)]
print(f"Dropped {before - len(df)} rows with extreme adr outliers")

# rows with zero total guests (adults+children+babies == 0) are data errors
before = len(df)
df = df[(df['adults'] + df['children'] + df['babies']) > 0]
print(f"Dropped {before - len(df)} rows with zero total guests")

# rows with zero total nights are also suspicious (never actually stayed)
before = len(df)
df = df[(df['stays_in_weekend_nights'] + df['stays_in_week_nights']) > 0]
print(f"Dropped {before - len(df)} rows with zero total nights")

# ------------------------------------------------------------------
# 4. Deduplicate
# ------------------------------------------------------------------
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows")

print(f"\nFinal cleaned shape: {df.shape}")
df.to_csv('dataset/hotel_booking_cleaned.csv', index=False)
print("Saved -> dataset/hotel_booking_cleaned.csv")


Original shape: (119390, 36)

Missing values before cleaning:
children         4
country        488
agent        16340
company     112593
dtype: int64

Dropped 1 rows with negative adr
Dropped 120 rows with extreme adr outliers
Dropped 180 rows with zero total guests
Dropped 645 rows with zero total nights
Dropped 32204 duplicate rows

Final cleaned shape: (86240, 29)
Saved -> dataset/hotel_booking_cleaned.csv


## 2. Exploratory Data Analysis

In [4]:
"""
Module 1: Hotel Booking Prediction
Step 2: Exploratory Data Analysis
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('dataset/hotel_booking_cleaned.csv')

# ------------------------------------------------------------------
# 1. Target distribution
# ------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df['is_canceled'].value_counts().plot(kind='bar', ax=ax[0], color=['#4C72B0', '#DD8452'])
ax[0].set_xticklabels(['Not Canceled', 'Canceled'], rotation=0)
ax[0].set_title('Booking Outcome Counts')
ax[0].set_ylabel('Count')

df.groupby('hotel')['is_canceled'].mean().plot(kind='bar', ax=ax[1], color=['#55A868', '#C44E52'])
ax[1].set_title('Cancellation Rate by Hotel Type')
ax[1].set_ylabel('Cancellation Rate')
ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('plots/01_target_distribution.png')
plt.close()

# ------------------------------------------------------------------
# 2. Lead time vs cancellation
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.kdeplot(data=df, x='lead_time', hue='is_canceled', fill=True, common_norm=False, ax=ax)
ax.set_title('Lead Time Distribution by Cancellation Status')
ax.set_xlim(0, 500)
plt.tight_layout()
plt.savefig('plots/02_lead_time_distribution.png')
plt.close()

# ------------------------------------------------------------------
# 3. Monthly booking & cancellation trend
# ------------------------------------------------------------------
month_order = ['January','February','March','April','May','June','July',
                'August','September','October','November','December']
monthly = df.groupby('arrival_date_month')['is_canceled'].agg(['count', 'mean']).reindex(month_order)

fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax2 = ax1.twinx()
ax1.bar(monthly.index, monthly['count'], color='#8CA0D7', alpha=0.6, label='Total Bookings')
ax2.plot(monthly.index, monthly['mean'], color='#C44E52', marker='o', linewidth=2, label='Cancellation Rate')
ax1.set_ylabel('Total Bookings')
ax2.set_ylabel('Cancellation Rate')
ax1.set_xticklabels(monthly.index, rotation=45)
ax1.set_title('Monthly Booking Volume & Cancellation Rate')
plt.tight_layout()
plt.savefig('plots/03_monthly_trend.png')
plt.close()

# ------------------------------------------------------------------
# 4. Categorical drivers of cancellation
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
cat_cols = ['deposit_type', 'customer_type', 'market_segment', 'distribution_channel']
for ax, col in zip(axes.flatten(), cat_cols):
    order = df.groupby(col)['is_canceled'].mean().sort_values(ascending=False).index
    sns.barplot(data=df, x=col, y='is_canceled', order=order, ax=ax, palette='viridis')
    ax.set_title(f'Cancellation Rate by {col}')
    ax.set_ylabel('Cancellation Rate')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('plots/04_categorical_drivers.png')
plt.close()

# ------------------------------------------------------------------
# 5. Correlation heatmap (numeric features)
# ------------------------------------------------------------------
numeric_cols = df.select_dtypes(include=[np.number]).columns
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(df[numeric_cols].corr(), cmap='coolwarm', center=0, annot=False, ax=ax)
ax.set_title('Correlation Heatmap - Numeric Features')
plt.tight_layout()
plt.savefig('plots/05_correlation_heatmap.png')
plt.close()

# ------------------------------------------------------------------
# 6. Key stats summary (for report)
# ------------------------------------------------------------------
summary = {
    'overall_cancellation_rate': df['is_canceled'].mean(),
    'city_hotel_cancel_rate': df[df.hotel=='City Hotel']['is_canceled'].mean(),
    'resort_hotel_cancel_rate': df[df.hotel=='Resort Hotel']['is_canceled'].mean(),
    'non_refund_cancel_rate': df[df.deposit_type=='Non Refund']['is_canceled'].mean(),
    'no_deposit_cancel_rate': df[df.deposit_type=='No Deposit']['is_canceled'].mean(),
    'avg_lead_time_canceled': df[df.is_canceled==1]['lead_time'].mean(),
    'avg_lead_time_not_canceled': df[df.is_canceled==0]['lead_time'].mean(),
}
print("Key EDA findings:")
for k, v in summary.items():
    print(f"  {k}: {v:.3f}")

print("\nPlots saved to plots/ directory")


C:\Users\windo\AppData\Local\Temp\ipykernel_21764\3998477901.py:56: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax1.set_xticklabels(monthly.index, rotation=45)
C:\Users\windo\AppData\Local\Temp\ipykernel_21764\3998477901.py:69: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x=col, y='is_canceled', order=order, ax=ax, palette='viridis')
C:\Users\windo\AppData\Local\Temp\ipykernel_21764\3998477901.py:69: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x=col, y='is_canceled', order=order, ax=ax, palette='viridis')
C:\Users\windo\AppData\Local\Temp\ipykernel_21764\3998477901.py:69: FutureWarnin

Key EDA findings:
  overall_cancellation_rate: 0.275
  city_hotel_cancel_rate: 0.300
  resort_hotel_cancel_rate: 0.235
  non_refund_cancel_rate: 0.946
  no_deposit_cancel_rate: 0.267
  avg_lead_time_canceled: 105.510
  avg_lead_time_not_canceled: 70.542

Plots saved to plots/ directory


## 3. Feature Engineering

In [5]:
"""
Module 1: Hotel Booking Prediction
Step 3: Feature Engineering
"""
import pandas as pd
import numpy as np

df = pd.read_csv('dataset/hotel_booking_cleaned.csv')

# ------------------------------------------------------------------
# 1. Stay & guest composition features
# ------------------------------------------------------------------
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df['is_family'] = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)

# ------------------------------------------------------------------
# 2. Booking behaviour features
# ------------------------------------------------------------------
df['room_type_mismatch'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)
df['total_previous_bookings'] = df['previous_cancellations'] + df['previous_bookings_not_canceled']
df['prev_cancellation_rate'] = df['previous_cancellations'] / (df['total_previous_bookings'] + 1)
df['is_returning_guest'] = df['is_repeated_guest']  # already binary, kept for clarity

# ------------------------------------------------------------------
# 3. Date-derived features
# ------------------------------------------------------------------
month_map = {m: i+1 for i, m in enumerate(
    ['January','February','March','April','May','June','July',
     'August','September','October','November','December'])}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map)

# season from month
def month_to_season(m):
    if m in [12, 1, 2]: return 'Winter'
    if m in [3, 4, 5]: return 'Spring'
    if m in [6, 7, 8]: return 'Summer'
    return 'Autumn'
df['season'] = df['arrival_month_num'].apply(month_to_season)

# ------------------------------------------------------------------
# 4. Price feature
# ------------------------------------------------------------------
df['adr_per_guest'] = df['adr'] / df['total_guests'].replace(0, 1)

# ------------------------------------------------------------------
# 5. Reduce high-cardinality categorical: country
# ------------------------------------------------------------------
top_countries = df['country'].value_counts().nlargest(15).index
df['country_grouped'] = df['country'].apply(lambda x: x if x in top_countries else 'Other')

# ------------------------------------------------------------------
# 6. Drop columns no longer needed post-engineering
# ------------------------------------------------------------------
drop_cols = ['country', 'arrival_date_month', 'arrival_date_year',
             'arrival_date_week_number', 'arrival_date_day_of_month',
             'reserved_room_type', 'assigned_room_type',
             'previous_cancellations', 'previous_bookings_not_canceled',
             'is_repeated_guest', 'stays_in_weekend_nights', 'stays_in_week_nights']
df = df.drop(columns=drop_cols)

print(f"Shape after feature engineering: {df.shape}")
print(f"\nNew columns added: total_nights, total_guests, is_family, room_type_mismatch,")
print(f"total_previous_bookings, prev_cancellation_rate, arrival_month_num, season,")
print(f"adr_per_guest, country_grouped")

df.to_csv('dataset/hotel_booking_featured.csv', index=False)
print("\nSaved -> data/hotel_booking_featured.csv")


Shape after feature engineering: (86240, 28)

New columns added: total_nights, total_guests, is_family, room_type_mismatch,
total_previous_bookings, prev_cancellation_rate, arrival_month_num, season,
adr_per_guest, country_grouped

Saved -> data/hotel_booking_featured.csv


## 4. Preprocessing

In [8]:
"""
Module 1: Hotel Booking Prediction
Step 4: Preprocessing (encoding, scaling, train/test split)
"""
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('dataset/hotel_booking_featured.csv')

TARGET = 'is_canceled'
y = df[TARGET]
X = df.drop(columns=[TARGET])

# ------------------------------------------------------------------
# 1. One-hot encode categorical columns
# ------------------------------------------------------------------
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {categorical_cols}")

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print(f"Shape after one-hot encoding: {X_encoded.shape}")

# ------------------------------------------------------------------
# 2. Train/test split (stratified — target has moderate imbalance)
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train target balance:\n{y_train.value_counts(normalize=True)}")

# ------------------------------------------------------------------
# 3. Scale numeric features (needed for ANN & Logistic Regression;
#    harmless for tree-based models)
# ------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------------
# 4. Save everything needed for modeling
# ------------------------------------------------------------------
np.save('dataset/X_train_scaled.npy', X_train_scaled)
np.save('dataset/X_test_scaled.npy', X_test_scaled)
X_train.to_csv('dataset/X_train_raw.csv', index=False)   # unscaled, for tree models
X_test.to_csv('dataset/X_test_raw.csv', index=False)
y_train.to_csv('dataset/y_train.csv', index=False)
y_test.to_csv('dataset/y_test.csv', index=False)
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(list(X_encoded.columns), 'models/feature_columns.pkl')

print("\nSaved preprocessed train/test dataset and scaler to dataset/ and models/")


Categorical columns to encode: ['hotel', 'meal', 'market_segment', 'distribution_channel', 'deposit_type', 'customer_type', 'season', 'country_grouped']
Shape after one-hot encoding: (86240, 58)

Train shape: (68992, 58), Test shape: (17248, 58)
Train target balance:
is_canceled
0    0.72533
1    0.27467
Name: proportion, dtype: float64

Saved preprocessed train/test dataset and scaler to dataset/ and models/


## 5. Baseline Models (Logistic Regression + Random Forest)

In [9]:
"""
Module 1: Hotel Booking Prediction
Step 5a: Baseline ML models (Logistic Regression + Random Forest)
These serve as the comparison benchmark against the deep learning ANN.
"""
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

X_train_scaled = np.load('dataset/X_train_scaled.npy')
X_test_scaled = np.load('dataset/X_test_scaled.npy')
X_train_raw = pd.read_csv('dataset/X_train_raw.csv')
X_test_raw = pd.read_csv('dataset/X_test_raw.csv')
y_train = pd.read_csv('dataset/y_train.csv').values.ravel()
y_test = pd.read_csv('dataset/y_test.csv').values.ravel()

results = {}

def evaluate(name, y_true, y_pred, y_proba):
    results[name] = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }
    print(f"\n--- {name} ---")
    for k, v in results[name].items():
        print(f"  {k}: {v:.4f}")
    return confusion_matrix(y_true, y_pred)

# ------------------------------------------------------------------
# 1. Logistic Regression (uses scaled features)
# ------------------------------------------------------------------
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
cm_lr = evaluate('Logistic Regression', y_test, lr_pred, lr_proba)

# ------------------------------------------------------------------
# 2. Random Forest (uses raw/unscaled features - tree models don't need scaling)
# ------------------------------------------------------------------
rf = RandomForestClassifier(n_estimators=300, max_depth=15, class_weight='balanced',
                              random_state=42, n_jobs=-1)
rf.fit(X_train_raw, y_train)
rf_pred = rf.predict(X_test_raw)
rf_proba = rf.predict_proba(X_test_raw)[:, 1]
cm_rf = evaluate('Random Forest', y_test, rf_pred, rf_proba)

# ------------------------------------------------------------------
# 3. Confusion matrices
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Canceled', 'Canceled'], yticklabels=['Not Canceled', 'Canceled'])
axes[0].set_title('Logistic Regression - Confusion Matrix')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Not Canceled', 'Canceled'], yticklabels=['Not Canceled', 'Canceled'])
axes[1].set_title('Random Forest - Confusion Matrix')
axes[1].set_ylabel('Actual'); axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('plots/06_baseline_confusion_matrices.png')
plt.close()

# ------------------------------------------------------------------
# 4. Feature importance (Random Forest)
# ------------------------------------------------------------------
importances = pd.Series(rf.feature_importances_, index=X_train_raw.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 6))
importances.plot(kind='barh', ax=ax, color='#55A868')
ax.invert_yaxis()
ax.set_title('Top 15 Feature Importances (Random Forest)')
plt.tight_layout()
plt.savefig('plots/07_feature_importance.png')
plt.close()

# ------------------------------------------------------------------
# 5. Save results & models
# ------------------------------------------------------------------
results_df = pd.DataFrame(results).T
results_df.to_csv('models/baseline_results.csv')
joblib.dump(lr, 'models/logistic_regression.pkl')
joblib.dump(rf, 'models/random_forest.pkl')

print("\n\nComparison table:")
print(results_df.round(4))
print("\nSaved models and results to models/")



--- Logistic Regression ---
  accuracy: 0.7498
  precision: 0.5297
  recall: 0.7948
  f1: 0.6357
  roc_auc: 0.8443

--- Random Forest ---
  accuracy: 0.7643
  precision: 0.5447
  recall: 0.8643
  f1: 0.6682
  roc_auc: 0.8866


Comparison table:
                     accuracy  precision  recall      f1  roc_auc
Logistic Regression    0.7498     0.5297  0.7948  0.6357   0.8443
Random Forest          0.7643     0.5447  0.8643  0.6682   0.8866

Saved models and results to models/


## 6. Deep Learning Model (ANN — requires TensorFlow, run in Colab)

In [10]:
"""
Module 1: Hotel Booking Prediction
Step 5b: Deep Learning Model (ANN with Keras/TensorFlow)

>>> RUN THIS IN GOOGLE COLAB OR A LOCAL ENVIRONMENT WITH TENSORFLOW INSTALLED <<<
(pip install tensorflow)

This script assumes the preprocessed .npy / .csv files from 04_preprocessing.py
are available in the same relative 'dataset/' and 'models/' folders.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)
np.random.seed(42)

# ------------------------------------------------------------------
# 1. Load preprocessed data
# ------------------------------------------------------------------
X_train = np.load('dataset/X_train_scaled.npy')
X_test = np.load('dataset/X_test_scaled.npy')
y_train = pd.read_csv('dataset/y_train.csv').values.ravel()
y_test = pd.read_csv('dataset/y_test.csv').values.ravel()

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

# handle class imbalance (~73/27 split)
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

# ------------------------------------------------------------------
# 2. Build the ANN
# ------------------------------------------------------------------
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.2),

    Dense(1, activation='sigmoid')   # binary output: cancel probability
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()

# ------------------------------------------------------------------
# 3. Train with early stopping
# ------------------------------------------------------------------
callbacks = [
    EarlyStopping(monitor='val_auc', mode='max', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# ------------------------------------------------------------------
# 4. Evaluate on test set
# ------------------------------------------------------------------
y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("\n--- ANN Test Set Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

# ------------------------------------------------------------------
# 5. Plot training curves
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy'); axes[1].legend()

axes[2].plot(history.history['auc'], label='Train AUC')
axes[2].plot(history.history['val_auc'], label='Val AUC')
axes[2].set_title('AUC'); axes[2].legend()
plt.tight_layout()
plt.savefig('plots/08_ann_training_curves.png')
plt.close()

# ------------------------------------------------------------------
# 6. Confusion matrix
# ------------------------------------------------------------------
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=['Not Canceled', 'Canceled'], yticklabels=['Not Canceled', 'Canceled'])
ax.set_title('ANN - Confusion Matrix')
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('plots/09_ann_confusion_matrix.png')
plt.close()

# ------------------------------------------------------------------
# 7. Save model
# ------------------------------------------------------------------
model.save('models/ann_booking_model.keras')
print("\nModel saved -> models/ann_booking_model.keras")


X_train: (68992, 58), X_test: (17248, 58)
Class weights: {0: np.float64(0.6893409535989768), 1: np.float64(1.8203693931398417)}


c:\Users\windo\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         7,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,689 (73.00 KB)

 Trainable params: 18,305 (71.50 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7002 - auc: 0.7879 - loss: 0.5559 - val_accuracy: 0.7427 - val_auc: 0.8663 - val_loss: 0.4660 - learning_rate: 0.0010
Epoch 2/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7412 - auc: 0.8462 - loss: 0.4771 - val_accuracy: 0.7527 - val_auc: 0.8733 - val_loss: 0.4523 - learning_rate: 0.0010
Epoch 3/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7492 - auc: 0.8553 - loss: 0.4620 - val_accuracy: 0.7572 - val_auc: 0.8754 - val_loss: 0.4482 - learning_rate: 0.0010
Epoch 4/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7564 - auc: 0.8618 - loss: 0.4530 - val_accuracy: 0.7638 - val_auc: 0.8786 - val_loss: 0.4415 - learning_rate: 0.0010
Epoch 5/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7626 - auc: 0.8668 - loss: 0.4458 - val_accuracy: 0.7719 - val_auc: 0.8808 - val_loss: 0.4360 - learning_rate: 0.0010
Epoch 6/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7684 - a